# Notebook 10_a: Skill Dictionary Adaptation & Expansion for Methodology 1.1 Profile Expansion: Rule- and Taxonomy-Based Skill Extraction

## Rule-Based Skill Database Expansion (Employment/LinkedIn Skills)

First, the existing skill vocabulary (`skills_vocab_raw.csv`) is expanded to include an additional, market-relevant skill dictionary from the Kaggle dataset “Employment Skills” (file: `linkedin skill`), similar to the dictionary setup in Cenikj et al. (2021). Cenikj uses four sources:
ESCO + LinkedIn + Employment + People Data Labs. ESCO is used here anyway; the other sources are not freely available. A suitable dataset on Employment (LinkedIn) Skills was found via Kaggle (Dec. 7, 2025: Disodia, https://www.kaggle.com/datasets/maneeshdisodia/employment-skills?select=linkedin+skill, file: linkedin skill), which is also used here in addition. The other data are not used, as no similar, freely accessible, and suitable datasets were found.

Other skill sources:
- The basis for the lexicon is `data/processed_external/skills_vocab_raw.csv` (Notebook 07) and already contains ESCO + BA (including synonyms).
- `Resume dataset (structured): 06_skills.csv` (approx. 227,000 skills, Ganesh, https://www.kaggle.com/datasets/suriyaganesh/resume-dataset-structured/data?select=01_people.csv) is not included (performance and noise risk). Optionally, the file is saved as an alternative export variant in the same format.

In [1]:
# Imports & Paths
from pathlib import Path
import pandas as pd
import re

PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

# Paths
DATA_RAW_EXTERNAL       = PROJECT_ROOT / "data" / "raw_external"
DATA_PROCESSED_EXTERNAL = PROJECT_ROOT / "data" / "processed_external"
DOCS_PATH               = PROJECT_ROOT / "docs"
DATA_PROCESSED_EXTERNAL.mkdir(parents=True, exist_ok=True)
SKILLS_VOCAB_RAW_PATH = DATA_PROCESSED_EXTERNAL / "skills_vocab_raw.csv"
LINKEDIN_SKILL_PATH = DATA_RAW_EXTERNAL / "standards" / "linkedin skill.txt"
RESUME06_SKILLS_PATH = DATA_RAW_EXTERNAL / "cvs" / "Resume dataset (structured)" / "06_skills.csv" # Optional

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SKILLS_VOCAB_RAW_PATH:", SKILLS_VOCAB_RAW_PATH, "exists:", SKILLS_VOCAB_RAW_PATH.exists())
print("LINKEDIN_SKILL_PATH:", LINKEDIN_SKILL_PATH, "exists:", LINKEDIN_SKILL_PATH.exists())
print("RESUME06_SKILLS_PATH:", RESUME06_SKILLS_PATH, "exists:", RESUME06_SKILLS_PATH.exists())

PROJECT_ROOT: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung
SKILLS_VOCAB_RAW_PATH: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\skills_vocab_raw.csv exists: True
LINKEDIN_SKILL_PATH: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\raw_external\standards\linkedin skill.txt exists: True
RESUME06_SKILLS_PATH: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\raw_external\cvs\Resume dataset (structured)\06_skills.csv exists: True


## 1. Load existing skill vocabulary:

In [2]:
df_vocab = pd.read_csv(SKILLS_VOCAB_RAW_PATH)
print("Vokabular geladen:", SKILLS_VOCAB_RAW_PATH)
print("Anzahl Einträge:", len(df_vocab))
display(df_vocab.head(3))

# Required columns Check like in Notebook 07
req = ["vocab_id", "label", "source_system", "source_id", "lang", "label_type"]
missing = [c for c in req if c not in df_vocab.columns]
assert not missing, f"skills_vocab_raw.csv missing columns: {missing}"

# Normalization
def normalize_label(text: str) -> str:
    if text is None:
        return ""
    t = str(text).strip().lower()
    t = re.sub(r"\s+", " ", t)
    return t

df_vocab["label"] = df_vocab["label"].astype(str).str.strip()
df_vocab = df_vocab[df_vocab["label"] != ""].copy()
df_vocab["label_norm"] = df_vocab["label"].map(normalize_label)

print("Nach Cleaning:", len(df_vocab))
print("label_norm missing:", df_vocab["label_norm"].isna().sum())

Vokabular geladen: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\skills_vocab_raw.csv
Anzahl Einträge: 111727


,vocab_id,label,source_system,source_id,lang,label_type
0,ESCO:http://data.europa.eu/esco/skill/0005c151...,manage musical staff,ESCO,http://data.europa.eu/esco/skill/0005c151-5b5a...,en,preferred
1,ESCO:http://data.europa.eu/esco/skill/00064735...,supervise correctional procedures,ESCO,http://data.europa.eu/esco/skill/00064735-8fad...,en,preferred
2,ESCO:http://data.europa.eu/esco/skill/000709ed...,apply anti-oppressive practices,ESCO,http://data.europa.eu/esco/skill/000709ed-2be5...,en,preferred


Nach Cleaning: 111727
label_norm missing: 0


## 2. LinkedIn Skills List

Import the Employment/LinkedIn skills list (Kaggle): check whether each line contains a skill name, perform some basic cleaning and normalization:

In [3]:
# Import LinkedIn/Employment Skills 
with open(LINKEDIN_SKILL_PATH, "r", encoding="utf-8", errors="ignore") as f:
    raw_lines = f.readlines()

df_linkedin_raw = pd.DataFrame({"label_raw": raw_lines})
df_linkedin_raw["label_raw"] = df_linkedin_raw["label_raw"].astype(str).str.strip()

# Remove empty lines
df_linkedin_raw = df_linkedin_raw[df_linkedin_raw["label_raw"] != ""].reset_index(drop=True)
print("Anzahl LinkedIn-Skills (inkl. Dubletten):", len(df_linkedin_raw))
display(df_linkedin_raw.head(10))

# Cleaning + Normalization
df_linkedin = df_linkedin_raw.copy()
df_linkedin["label"] = df_linkedin["label_raw"].astype(str).str.strip()
df_linkedin["label_norm"] = df_linkedin["label"].map(normalize_label)

# Deduplication on label_norm
before = len(df_linkedin)
df_linkedin = df_linkedin.drop_duplicates(subset=["label_norm"]).reset_index(drop=True)
after = len(df_linkedin)
print("Einzigartige LinkedIn-Labels:", after, "(vorher", before, ")")
display(df_linkedin.head(10))

Anzahl LinkedIn-Skills (inkl. Dubletten): 36944


,label_raw
0,(ISC)2
1,.NET
2,.NET CLR
3,.NET Compact Framework
4,.NET Framework
5,.NET Remoting
6,.Net Core
7,.com
8,.htaccess
9,1-4 Units


Einzigartige LinkedIn-Labels: 36944 (vorher 36944 )


,label_raw,label,label_norm
0,(ISC)2,(ISC)2,(isc)2
1,.NET,.NET,.net
2,.NET CLR,.NET CLR,.net clr
3,.NET Compact Framework,.NET Compact Framework,.net compact framework
4,.NET Framework,.NET Framework,.net framework
5,.NET Remoting,.NET Remoting,.net remoting
6,.Net Core,.Net Core,.net core
7,.com,.com,.com
8,.htaccess,.htaccess,.htaccess
9,1-4 Units,1-4 Units,1-4 units


-> a column containing skill entries, one skill per row. A total of 36,944 entries

## 3. Normalization

for later deduplication

In [4]:
df_linkedin["label_norm"] = df_linkedin["label"].map(normalize_label)

# If there are case variants of the same skill
before_norm = len(df_linkedin)
df_linkedin = df_linkedin.drop_duplicates(subset=["label_norm"]).reset_index(drop=True)
after_norm = len(df_linkedin)

print(f"Nach Normalisierung & Deduplikation nach label_norm: {after_norm} (vorher {before_norm})")
df_linkedin.head(10)

Nach Normalisierung & Deduplikation nach label_norm: 36944 (vorher 36944)


,label_raw,label,label_norm
0,(ISC)2,(ISC)2,(isc)2
1,.NET,.NET,.net
2,.NET CLR,.NET CLR,.net clr
3,.NET Compact Framework,.NET Compact Framework,.net compact framework
4,.NET Framework,.NET Framework,.net framework
5,.NET Remoting,.NET Remoting,.net remoting
6,.Net Core,.Net Core,.net core
7,.com,.com,.com
8,.htaccess,.htaccess,.htaccess
9,1-4 Units,1-4 Units,1-4 units


## 4. Comparison with the existing skill database:

In [5]:
# Vocabulary with normalized labels
df_vocab_norm = df_vocab.copy()

# LinkedIn skills that haven't appeared in the vocabulary so far
df_linkedin_new = (
    df_linkedin
    .merge(
        df_vocab_norm[["label_norm"]].drop_duplicates(),
        on="label_norm",
        how="left",
        indicator=True
    )
)

df_linkedin_new = df_linkedin_new[df_linkedin_new["_merge"] == "left_only"].drop(columns=["_merge"]).reset_index(drop=True) # Only skills that do not appear in ESCO + BA

print("Neue LinkedIn-Skills, die noch nicht in ESCO+BA vorkommen:", len(df_linkedin_new))
display(df_linkedin_new.head(10))

Neue LinkedIn-Skills, die noch nicht in ESCO+BA vorkommen: 34874


,label_raw,label,label_norm
0,(ISC)2,(ISC)2,(isc)2
1,.NET,.NET,.net
2,.NET CLR,.NET CLR,.net clr
3,.NET Compact Framework,.NET Compact Framework,.net compact framework
4,.NET Remoting,.NET Remoting,.net remoting
5,.Net Core,.Net Core,.net core
6,.com,.com,.com
7,.htaccess,.htaccess,.htaccess
8,1-4 Units,1-4 Units,1-4 units
9,1-Wire,1-Wire,1-wire


Create DB entries for the dataset here; minor renaming; same (column) format:

In [6]:
def is_noise_linkedin(label_norm: str) -> bool: # label as a noise tag for LinkedIn posts
    if not isinstance(label_norm, str):
        return True
    t = label_norm.strip()
    if t == "":
        return True
    if len(t) <= 2:
        return True
    if t.startswith("."):
        return True
    if re.fullmatch(r"[0-9\W_]+", t):
        return True
    if re.fullmatch(r"v?\d+(\.\d+){1,3}", t):
        return True
    return False

vocab_rows_linkedin = []
for _, row in df_linkedin_new.iterrows():
    label = row["label"]
    label_norm = row["label_norm"]
    vocab_rows_linkedin.append({
        "vocab_id": f"LINKEDIN:{label_norm}", # unique key
        "label": label,
        "source_system": "LINKEDIN",
        "source_id": "linkedin_skilllist",
        "lang": "en",
        "label_type": "linkedin_skill",
        "label_norm": label_norm,
        "is_noise": is_noise_linkedin(label_norm),
    })

df_vocab_linkedin = pd.DataFrame(vocab_rows_linkedin)
print("Anzahl neuer Vokabel-Einträge aus LinkedIn:", len(df_vocab_linkedin))
display(df_vocab_linkedin.head(10))

Anzahl neuer Vokabel-Einträge aus LinkedIn: 34874


,vocab_id,label,source_system,source_id,lang,label_type,label_norm,is_noise
0,LINKEDIN:(isc)2,(ISC)2,LINKEDIN,linkedin_skilllist,en,linkedin_skill,(isc)2,False
1,LINKEDIN:.net,.NET,LINKEDIN,linkedin_skilllist,en,linkedin_skill,.net,True
2,LINKEDIN:.net clr,.NET CLR,LINKEDIN,linkedin_skilllist,en,linkedin_skill,.net clr,True
3,LINKEDIN:.net compact framework,.NET Compact Framework,LINKEDIN,linkedin_skilllist,en,linkedin_skill,.net compact framework,True
4,LINKEDIN:.net remoting,.NET Remoting,LINKEDIN,linkedin_skilllist,en,linkedin_skill,.net remoting,True
5,LINKEDIN:.net core,.Net Core,LINKEDIN,linkedin_skilllist,en,linkedin_skill,.net core,True
6,LINKEDIN:.com,.com,LINKEDIN,linkedin_skilllist,en,linkedin_skill,.com,True
7,LINKEDIN:.htaccess,.htaccess,LINKEDIN,linkedin_skilllist,en,linkedin_skill,.htaccess,True
8,LINKEDIN:1-4 units,1-4 Units,LINKEDIN,linkedin_skilllist,en,linkedin_skill,1-4 units,False
9,LINKEDIN:1-wire,1-Wire,LINKEDIN,linkedin_skilllist,en,linkedin_skill,1-wire,False


## 5. Final Skill Vocabulary for Methodology 1.1 + Export

The “skills_vocab_with_linkedin.csv” file has been properly generated and updated.

Merge with existing vocabulary and save:

In [7]:
df_vocab_extended = df_vocab.copy()
df_vocab_extended["is_noise"] = False  # Base no Noise-Flagging

In [8]:
# Merge Base + Linkedin
df_vocab_extended = pd.concat([df_vocab_extended, df_vocab_linkedin], ignore_index=True)

# Cleanup & Deduplication (at the label level)
before = len(df_vocab_extended)
df_vocab_extended = (
    df_vocab_extended
    .drop_duplicates(subset=["source_system","source_id","label_norm"])
    .reset_index(drop=True)
)
after = len(df_vocab_extended)
print("Gesamtgröße nach Deduplikation:", after, "(vorher", before, ")")
print(df_vocab_extended["source_system"].value_counts())

Gesamtgröße nach Deduplikation: 126277 (vorher 146601 )
source_system
BA          77464
LINKEDIN    34874
ESCO        13939
Name: count, dtype: int64


Result: New LinkedIn skills that haven't appeared yet: 34,874

36,944 total
− 2,070 duplicates
= 34,874 new skills

Checking for duplicates with the same vocab_id:

In [9]:
dup_mask = df_vocab_extended["vocab_id"].duplicated(keep=False)
print("Duplicate vocab_id rows:", dup_mask.sum())
display(df_vocab_extended.loc[dup_mask, ["vocab_id","source_system","source_id","label_type","lang","label","label_norm"]].head(30))

Duplicate vocab_id rows: 108


,vocab_id,source_system,source_id,label_type,lang,label,label_norm
14171,BA:K 0003:label,BA,K 0003,ba_label,de,Gartengestaltung nach Permakulturprinzipien,gartengestaltung nach permakulturprinzipien
14173,BA:K 0003:label,BA,K 0003,ba_label,de,Gartenbau,gartenbau
14440,BA:K 0004:label,BA,K 0004,ba_label,de,Agrarwirtschaftlicher Zweig,agrarwirtschaftlicher zweig
14442,BA:K 0004:label,BA,K 0004,ba_label,de,Agrarwirtschaft,agrarwirtschaft
14443,BA:K 0004:label,BA,K 0004,ba_label,de,Agrarwesen,agrarwesen
14444,BA:K 0004:label,BA,K 0004,ba_label,de,Landwirtschaft,landwirtschaft
14849,BA:K 0005:label,BA,K 0005,ba_label,de,Artgerechte Tierhaltung,artgerechte tierhaltung
14851,BA:K 0005:label,BA,K 0005,ba_label,de,Agrarwesen,agrarwesen
14852,BA:K 0005:label,BA,K 0005,ba_label,de,"Landwirtschaftliche Tierhaltung, Tierzucht","landwirtschaftliche tierhaltung, tierzucht"
20422,BA:K 0106:label,BA,K 0106,ba_label,de,Schmuck,schmuck


Rebuild vocab_id, then duplicates = 0

In [10]:
# vocab_id new
def make_vocab_id(row) -> str:
    src = str(row.get("source_system", "")).upper().strip()
    sid = str(row.get("source_id", "")).strip()
    ltype = str(row.get("label_type", "")).lower().strip()
    lnorm = str(row.get("label_norm", "")).strip()

    # LINKEDIN: source_id is always the same
    if src == "LINKEDIN":
        return f"LINKEDIN:{lnorm}"

    # ESCO/BA: source_id is useful; label_type + label_norm ensure uniqueness
    if src == "ESCO":
        return f"ESCO:{sid}:{ltype}:{lnorm}"

    if src == "BA":
        return f"BA:{sid}:{ltype}:{lnorm}"

    return f"{src}:{sid}:{ltype}:{lnorm}"

df_vocab_extended["vocab_id"] = df_vocab_extended.apply(make_vocab_id, axis=1) # Regenerate vocab_id
df_vocab_extended = df_vocab_extended.drop_duplicates(subset=["vocab_id"]).reset_index(drop=True) # drop duplicates just to be safe

# Check
dup = df_vocab_extended["vocab_id"].duplicated().sum()
print("Duplicate vocab_id:", dup)
assert dup == 0

Duplicate vocab_id: 0


The final skill vocabulary is fully generated and exported. It serves as a central label lexicon for rule- and lexicon-based skill extraction (in Notebook 10b):
- `skills_vocab_with_linkedin.csv` remains a label lexicon (rows = labels/synonyms/variants from ESCO, BA, and LinkedIn)
- Additionally, a concept level is introduced:
  - `skill_id` = stable concept ID (join key for the entire Methodology 1.1)
  - Each vocabulary row is uniquely assigned to a `skill_id`
- Potentially generic LinkedIn skills are not removed but are marked with `is_noise`

The distinction between the label level (`vocab_id`) and the concept level (`skill_id`) follows ontology-based competency models, in which multiple linguistic representations are assigned to a stable competency concept (see Calhau et al., 2021; Chiarello et al., 2018).

In [11]:
OUT_VOCAB = DATA_PROCESSED_EXTERNAL / "skills_vocab_with_linkedin.csv"
df_vocab_extended.to_csv(OUT_VOCAB, index=False, encoding="utf-8")
print("Saved:", OUT_VOCAB, "rows:", len(df_vocab_extended))

Saved: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\skills_vocab_with_linkedin.csv rows: 126277


Utility functions (normalization + derive skill_id):

More robust, so that ESCO/BA/LinkedIn are unambiguous.
- LinkedIn: LINKEDIN:<label_norm>
- ESCO: ESCO:<source_id> (source_id is the URI)
- BA: BA:<source_id> or fallback to vocab_id if source_id is not valid

In [12]:
def derive_skill_id(row) -> str:
    src = str(row.get("source_system", "")).upper().strip()
    source_id = str(row.get("source_id", "")).strip()
    label_norm = str(row.get("label_norm", "")).strip()

    if src == "LINKEDIN":
        return f"LINKEDIN:{label_norm}" if label_norm else "LINKEDIN:UNKNOWN"

    if src == "ESCO":
        # Notebook 07: source_id = skill_uri
        return f"ESCO:{source_id}" if source_id else "ESCO:UNKNOWN"

    if src == "BA":
        # Notebook 07: source_id = BA code
        return f"BA:{source_id}" if source_id else "BA:UNKNOWN"

    return f"{src}:{label_norm}" if src else f"UNKNOWN:{label_norm}"

Apply to the notebook after the merge (add skill_id):

In [13]:
df_vocab_extended["skill_id"] = df_vocab_extended.apply(derive_skill_id, axis=1)

# Check: skill_id must not be missing
print("skill_id missing:", df_vocab_extended["skill_id"].isna().sum())
print("unique skill_id:", df_vocab_extended["skill_id"].nunique())
display(df_vocab_extended[["vocab_id","skill_id","label","source_system","source_id","lang","label_type"]].head(10))

skill_id missing: 0
unique skill_id: 58469


,vocab_id,skill_id,label,source_system,source_id,lang,label_type
0,ESCO:http://data.europa.eu/esco/skill/0005c151...,ESCO:http://data.europa.eu/esco/skill/0005c151...,manage musical staff,ESCO,http://data.europa.eu/esco/skill/0005c151-5b5a...,en,preferred
1,ESCO:http://data.europa.eu/esco/skill/00064735...,ESCO:http://data.europa.eu/esco/skill/00064735...,supervise correctional procedures,ESCO,http://data.europa.eu/esco/skill/00064735-8fad...,en,preferred
2,ESCO:http://data.europa.eu/esco/skill/000709ed...,ESCO:http://data.europa.eu/esco/skill/000709ed...,apply anti-oppressive practices,ESCO,http://data.europa.eu/esco/skill/000709ed-2be5...,en,preferred
3,ESCO:http://data.europa.eu/esco/skill/0007bdc2...,ESCO:http://data.europa.eu/esco/skill/0007bdc2...,control compliance of railway vehicles regulat...,ESCO,http://data.europa.eu/esco/skill/0007bdc2-dd15...,en,preferred
4,ESCO:http://data.europa.eu/esco/skill/00090cc1...,ESCO:http://data.europa.eu/esco/skill/00090cc1...,identify available services,ESCO,http://data.europa.eu/esco/skill/00090cc1-1f27...,en,preferred
5,ESCO:http://data.europa.eu/esco/skill/000bb1e4...,ESCO:http://data.europa.eu/esco/skill/000bb1e4...,perform toxicological studies,ESCO,http://data.europa.eu/esco/skill/000bb1e4-89f0...,en,preferred
6,ESCO:http://data.europa.eu/esco/skill/000c94d2...,ESCO:http://data.europa.eu/esco/skill/000c94d2...,ensure coquille uniformity,ESCO,http://data.europa.eu/esco/skill/000c94d2-2a2e...,en,preferred
7,ESCO:http://data.europa.eu/esco/skill/000f1d3d...,ESCO:http://data.europa.eu/esco/skill/000f1d3d...,Haskell,ESCO,http://data.europa.eu/esco/skill/000f1d3d-220f...,en,preferred
8,ESCO:http://data.europa.eu/esco/skill/001115fb...,ESCO:http://data.europa.eu/esco/skill/001115fb...,show initiative,ESCO,http://data.europa.eu/esco/skill/001115fb-569f...,en,preferred
9,ESCO:http://data.europa.eu/esco/skill/001d46db...,ESCO:http://data.europa.eu/esco/skill/001d46db...,train staff to reduce food waste,ESCO,http://data.europa.eu/esco/skill/001d46db-035e...,en,preferred


Checks:

In [14]:
# Skill ID Coverage
print("Rows:", len(df_vocab_extended))
print("skill_id missing:", df_vocab_extended["skill_id"].isna().sum())
print("label_norm missing:", df_vocab_extended["label_norm"].eq("").sum())
print("\nUnique skill_id:", df_vocab_extended["skill_id"].nunique())
print("Unique vocab_id:", df_vocab_extended["vocab_id"].nunique())

Rows: 126277
skill_id missing: 0
label_norm missing: 0

Unique skill_id: 58469
Unique vocab_id: 126277


Noise Analysis for LinkedIn: Since the LinkedIn dataset contains many entries with very short or generic skills

In [15]:
# Problematic LinkedIn Strings: Entries such as .com, .net, 100-105, etc., are skills, but they create noise
df_li = df_vocab_extended[df_vocab_extended["source_system"].str.upper() == "LINKEDIN"].copy()

too_short = df_li[df_li["label_norm"].str.len() <= 2] # Very short entries
dot_starts = df_li[df_li["label_norm"].str.match(r"^\.")] # start with a period
mostly_digits = df_li[df_li["label_norm"].str.match(r"^[0-9\-\+\.]+$")] # many characters

print("LinkedIn <=2 chars:", len(too_short))
print("LinkedIn starts with dot:", len(dot_starts))
print("LinkedIn mostly digits/symbols:", len(mostly_digits))

df_li_ext = df_vocab_extended[df_vocab_extended["source_system"].str.upper() == "LINKEDIN"].copy()
print("LinkedIn rows:", len(df_li_ext))
print("LinkedIn noise flagged:", int(df_li_ext["is_noise"].sum()))
display(df_li_ext[df_li_ext["is_noise"]].head(25))

LinkedIn <=2 chars: 147
LinkedIn starts with dot: 7
LinkedIn mostly digits/symbols: 11
LinkedIn rows: 34874
LinkedIn noise flagged: 166


,vocab_id,label,source_system,source_id,lang,label_type,label_norm,is_noise,skill_id
91404,LINKEDIN:.net,.NET,LINKEDIN,linkedin_skilllist,en,linkedin_skill,.net,True,LINKEDIN:.net
91405,LINKEDIN:.net clr,.NET CLR,LINKEDIN,linkedin_skilllist,en,linkedin_skill,.net clr,True,LINKEDIN:.net clr
91406,LINKEDIN:.net compact framework,.NET Compact Framework,LINKEDIN,linkedin_skilllist,en,linkedin_skill,.net compact framework,True,LINKEDIN:.net compact framework
91407,LINKEDIN:.net remoting,.NET Remoting,LINKEDIN,linkedin_skilllist,en,linkedin_skill,.net remoting,True,LINKEDIN:.net remoting
91408,LINKEDIN:.net core,.Net Core,LINKEDIN,linkedin_skilllist,en,linkedin_skill,.net core,True,LINKEDIN:.net core
91409,LINKEDIN:.com,.com,LINKEDIN,linkedin_skilllist,en,linkedin_skill,.com,True,LINKEDIN:.com
91410,LINKEDIN:.htaccess,.htaccess,LINKEDIN,linkedin_skilllist,en,linkedin_skill,.htaccess,True,LINKEDIN:.htaccess
91415,LINKEDIN:100-105,100-105,LINKEDIN,linkedin_skilllist,en,linkedin_skill,100-105,True,LINKEDIN:100-105
91426,LINKEDIN:11781,11781,LINKEDIN,linkedin_skilllist,en,linkedin_skill,11781,True,LINKEDIN:11781
91435,LINKEDIN:1709,1709,LINKEDIN,linkedin_skilllist,en,linkedin_skill,1709,True,LINKEDIN:1709


Filter option:

In [16]:
df_vocab_extended_filtered = df_vocab_extended.copy()
df_vocab_extended_filtered = df_vocab_extended_filtered[
    ~((df_vocab_extended_filtered["source_system"].str.upper()=="LINKEDIN") & (df_vocab_extended_filtered["is_noise"]))].copy()

print("Filtered vocab rows:", len(df_vocab_extended_filtered))
print(df_vocab_extended_filtered["source_system"].value_counts())

Filtered vocab rows: 126111
source_system
BA          77464
LINKEDIN    34708
ESCO        13939
Name: count, dtype: int64


## 6. Skills-Concepts Table (1 row per skill_id)

Later in 10_b, always join skill_id and display_label. For display_label: If label_type == “preferred” exists—that is, “preferred” (per skill_id)—then take the most frequent or first label entry

In [17]:
# Preferred labels by skill_id
pref = (
    df_vocab_extended[df_vocab_extended["label_type"].astype(str).str.lower().eq("preferred")]
    .sort_values(["skill_id", "lang"])  # Stabilization
    .drop_duplicates("skill_id")
    [["skill_id", "label"]]
    .rename(columns={"label": "display_label"})
)

fallback = (
    df_vocab_extended
    .sort_values(["skill_id", "lang"])
    .drop_duplicates("skill_id")
    [["skill_id", "label"]]
    .rename(columns={"label": "display_label"})
)

skills_concepts = (
    fallback.merge(pref, on="skill_id", how="left", suffixes=("_fallback",""))
)

def build_skills_concepts(dfv: pd.DataFrame) -> pd.DataFrame:
    df = dfv.copy()
    df["label_type_norm"] = df["label_type"].astype(str).str.lower()

    # Prioritization: preferred > ba_label > linkedin_skill > alt > ba_synonym_tech > ba_synonym
    prio = {
        "preferred": 0,
        "ba_label": 1,
        "linkedin_skill": 2,
        "alt": 3,
        "ba_synonym_tech": 4,
        "ba_synonym": 5,
    }
    df["prio"] = df["label_type_norm"].map(lambda x: prio.get(x, 99))
    df["label_len"] = df["label"].astype(str).str.len()

    df = df.sort_values(["prio","label_len","label"])

    concepts = (
        df.drop_duplicates(subset=["skill_id"])[["skill_id","label","source_system","lang"]]
        .rename(columns={"label":"display_label"})
        .reset_index(drop=True)
    )
    return concepts

# Create concepts for both versions
skills_concepts_full = build_skills_concepts(df_vocab_extended)
skills_concepts_filtered = build_skills_concepts(df_vocab_extended_filtered)

print("Concepts full:", len(skills_concepts_full))
print("Concepts filtered:", len(skills_concepts_filtered))
display(skills_concepts_filtered.head(10))

Concepts full: 58469
Concepts filtered: 58303


,skill_id,display_label,source_system,lang
0,ESCO:http://data.europa.eu/esco/skill/51586df8...,R,ESCO,en
1,ESCO:http://data.europa.eu/esco/skill/4c016b68...,C#,ESCO,en
2,ESCO:http://data.europa.eu/esco/skill/58d7a289...,APL,ESCO,en
3,ESCO:http://data.europa.eu/esco/skill/b633eb55...,C++,ESCO,en
4,ESCO:http://data.europa.eu/esco/skill/e5d1f825...,CSS,ESCO,en
5,ESCO:http://data.europa.eu/esco/skill/3cee858e...,DB2,ESCO,en
6,ESCO:http://data.europa.eu/esco/skill/1b7c716e...,MDX,ESCO,en
7,ESCO:http://data.europa.eu/esco/skill/4350c38d...,PHP,ESCO,en
8,ESCO:http://data.europa.eu/esco/skill/598de5b0...,SQL,ESCO,en
9,ESCO:http://data.europa.eu/esco/skill/ce26e71f...,iOS,ESCO,en


In [18]:
# checks
print(skills_concepts_full["source_system"].value_counts().head(10))
print(skills_concepts_filtered["source_system"].value_counts().head(10))

source_system
LINKEDIN    34874
ESCO        13939
BA           9656
Name: count, dtype: int64
source_system
LINKEDIN    34708
ESCO        13939
BA           9656
Name: count, dtype: int64


## 7. In addition, BA entries were heuristically labeled as “leaf skills” versus “groupings” in order to give preference to genuine skill leaves during extraction:

In [19]:
# Define Leafs, since BA competencies include groups as well as skills
def extract_ba_code_from_row(row) -> str: # Extract the BA code from skill_id, source_id, or vocab_id
    # Candidates in a logical order
    candidates = [
        row.get("skill_id", ""),
        row.get("source_id", ""),
        row.get("vocab_id", "")
    ]
    for c in candidates:
        s = str(c).strip()
        if not s or s.lower() == "nan":
            continue

        # If, for example, skill_id is “BA:K 0001-007,” remove the prefix
        if s.startswith("BA:"):
            s = s[3:].strip()

        # Pattern: The string contains “K 0001-007” or “K 0001”
        m = re.search(r"\bK\s*\d{2,4}(?:-\d{3})?\b", s)
        if m:
            code = m.group(0)
            # Normalize whitespace: “K0001-007” -> “K 0001-007”
            code = re.sub(r"\s+", " ", code.strip())
            code = re.sub(r"^K(?=\d)", "K ", code)
            return code

    return ""

def ba_is_leaf_code(ba_code: str) -> bool: # Leaf if format: K ####-###
    if not ba_code:
        return False
    return bool(re.fullmatch(r"K\s*\d{2,4}-\d{3}", ba_code))

# apply
df_vocab_extended = df_vocab_extended.copy()

mask_ba = df_vocab_extended["source_system"].astype(str).str.upper().eq("BA")

df_vocab_extended.loc[mask_ba, "ba_code_norm"] = (
    df_vocab_extended.loc[mask_ba].apply(extract_ba_code_from_row, axis=1)
)

df_vocab_extended.loc[mask_ba, "ba_is_leaf"] = (
    df_vocab_extended.loc[mask_ba, "ba_code_norm"].map(ba_is_leaf_code)
)

print("BA rows:", mask_ba.sum())
print("BA code extracted (non-empty):", (df_vocab_extended.loc[mask_ba, "ba_code_norm"] != "").sum())
print("BA leaf candidates:", df_vocab_extended.loc[mask_ba, "ba_is_leaf"].sum())
print("BA group-like:", (~df_vocab_extended.loc[mask_ba, "ba_is_leaf"].fillna(False)).sum())

# Sample Output
display(df_vocab_extended.loc[mask_ba, ["skill_id","vocab_id","source_id","ba_code_norm","ba_is_leaf","label"]].sample(20, random_state=42))

BA rows: 77464
BA code extracted (non-empty): 28564
BA leaf candidates: 28286
BA group-like: 49178


C:\Users\sigle\AppData\Local\Temp\ipykernel_26808\228656020.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  print("BA group-like:", (~df_vocab_extended.loc[mask_ba, "ba_is_leaf"].fillna(False)).sum())


,skill_id,vocab_id,source_id,ba_code_norm,ba_is_leaf,label
68595,BA:K 090202-044,BA:K 090202-044:ba_synonym:nagelkosmetik,K 090202-044,,False,Nagelkosmetik
79178,BA:K 100104-029,BA:K 100104-029:ba_synonym:reportagen,K 100104-029,,False,Reportagen
46980,BA:K 070104-071,BA:K 070104-071:ba_synonym:samplitude musik-/a...,K 070104-071,,False,Samplitude Musik-/Audiosoftware
24524,BA:K 011102-021,BA:K 011102-021:ba_synonym_tech:formengiesserei,K 011102-021,,False,FORMENGIESSEREI
54339,BA:K 070403-021,BA:K 070403-021:ba_synonym_tech:xconcept,K 070403-021,,False,XCONCEPT
75675,BA:K 100005-007,BA:K 100005-007:ba_synonym_tech:ebass,K 100005-007,,False,EBASS
44938,BA:K 070102-058,BA:K 070102-058:ba_label:simulationssoftware m...,K 070102-058,,False,Simulationssoftware Maxwell
87985,BA:K 1401-060,BA:K 1401-060:ba_synonym_tech:schichtfuehrerin...,K 1401-060,K 1401-060,True,SCHICHTFUEHRERINWERKFEUERWEHR
90937,BA:K 16-015,BA:K 16-015:ba_synonym_tech:bibliothekoeffentl...,K 16-015,K 16-015,True,BIBLIOTHEKOEFFENTLICHE
44327,BA:K 070102-006,BA:K 070102-006:ba_synonym:designer (designsof...,K 070102-006,,False,Designer (Designsoftware)


In [20]:
import numpy as np

def ba_is_group_code(ba_code: str) -> bool:  # Groups: z.B. K 00, K 0001
    if not ba_code:
        return False
    return bool(re.fullmatch(r"K\s*\d{2,4}", ba_code))

mask_ba = df_vocab_extended["source_system"].astype(str).str.upper().eq("BA")

# Flags
df_vocab_extended.loc[mask_ba, "ba_has_code"] = df_vocab_extended.loc[mask_ba, "ba_code_norm"].ne("")
df_vocab_extended.loc[mask_ba, "ba_is_leaf"] = df_vocab_extended.loc[mask_ba, "ba_code_norm"].map(ba_is_leaf_code)
df_vocab_extended.loc[mask_ba, "ba_is_group"] = df_vocab_extended.loc[mask_ba, "ba_code_norm"].map(ba_is_group_code)

# Class
df_vocab_extended.loc[mask_ba, "ba_class"] = np.select(
    [
        df_vocab_extended.loc[mask_ba, "ba_is_leaf"].fillna(False),
        df_vocab_extended.loc[mask_ba, "ba_is_group"].fillna(False),
        ~df_vocab_extended.loc[mask_ba, "ba_has_code"].fillna(False),
    ],
    ["leaf", "group", "unknown"],
    default="other"
)

print("BA rows:", mask_ba.sum())
print("BA has code:", df_vocab_extended.loc[mask_ba, "ba_has_code"].sum())
print(df_vocab_extended.loc[mask_ba, "ba_class"].value_counts())

# Show explicit examples of "unknown" and "group"
display(df_vocab_extended.loc[mask_ba & (df_vocab_extended["ba_class"]=="group"), ["skill_id","vocab_id","source_id","ba_code_norm","label"]].head(15))
display(df_vocab_extended.loc[mask_ba & (df_vocab_extended["ba_class"]=="unknown"), ["skill_id","vocab_id","source_id","ba_code_norm","label","label_type"]].head(15))

BA rows:

C:\Users\sigle\AppData\Local\Temp\ipykernel_26808\944077975.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_vocab_extended.loc[mask_ba, "ba_is_leaf"].fillna(False),
C:\Users\sigle\AppData\Local\Temp\ipykernel_26808\944077975.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_vocab_extended.loc[mask_ba, "ba_is_group"].fillna(False),
C:\Users\sigle\AppData\Local\Temp\ipykernel_26808\944077975.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_obj

 77464
BA has code: 28564
ba_class
unknown    48900
leaf       28286
group        278
Name: count, dtype: int64


,skill_id,vocab_id,source_id,ba_code_norm,label
13939,BA:K 00,"BA:K 00:ba_label:land-, forstwirtschaft, garte...",K 00,K 00,"Land-, Forstwirtschaft, Gartenbau"
13940,BA:K 00,BA:K 00:ba_synonym_tech:landforstwirtschaftgar...,K 00,K 00,LANDFORSTWIRTSCHAFTGARTENBAU
13941,BA:K 0001,BA:K 0001:ba_label:floristik,K 0001,K 0001,Floristik
14011,BA:K 0002,"BA:K 0002:ba_label:forstwirtschaft, jagd",K 0002,K 0002,"Forstwirtschaft, Jagd"
14012,BA:K 0002,BA:K 0002:ba_synonym_tech:forstwirtschaftjagd,K 0002,K 0002,FORSTWIRTSCHAFTJAGD
14171,BA:K 0003,BA:K 0003:ba_label:gartengestaltung nach perma...,K 0003,K 0003,Gartengestaltung nach Permakulturprinzipien
14172,BA:K 0003,BA:K 0003:ba_synonym_tech:gartengestaltungnach...,K 0003,K 0003,GARTENGESTALTUNGNACHPERMAKULTURPRINZIPIEN
14173,BA:K 0003,BA:K 0003:ba_label:gartenbau,K 0003,K 0003,Gartenbau
14440,BA:K 0004,BA:K 0004:ba_label:agrarwirtschaftlicher zweig,K 0004,K 0004,Agrarwirtschaftlicher Zweig
14441,BA:K 0004,BA:K 0004:ba_synonym_tech:agrarwirtschaftliche...,K 0004,K 0004,AGRARWIRTSCHAFTLICHERZWEIG


,skill_id,vocab_id,source_id,ba_code_norm,label,label_type
15405,BA:K 010000,BA:K 010000:ba_label:baustoffherstellung,K 010000,,Baustoffherstellung,ba_label
15406,BA:K 010000-000,BA:K 010000-000:ba_label:asphaltmischungen her...,K 010000-000,,Asphaltmischungen herstellen,ba_label
15407,BA:K 010000-000,BA:K 010000-000:ba_synonym:asphaltmischungsher...,K 010000-000,,Asphaltmischungsherstellung,ba_synonym
15408,BA:K 010000-000,BA:K 010000-000:ba_synonym_tech:asphaltmischun...,K 010000-000,,ASPHALTMISCHUNGENHERSTELLEN,ba_synonym_tech
15409,BA:K 010000-000,BA:K 010000-000:ba_synonym:asphaltmischungen,K 010000-000,,Asphaltmischungen,ba_synonym
15410,BA:K 010000-000,BA:K 010000-000:ba_synonym:asphalttechnik,K 010000-000,,Asphalttechnik,ba_synonym
15411,BA:K 010000-000,BA:K 010000-000:ba_synonym:asphaltherstellung,K 010000-000,,Asphaltherstellung,ba_synonym
15412,BA:K 010000-001,BA:K 010000-001:ba_label:betonfertigteile hers...,K 010000-001,,Betonfertigteile herstellen,ba_label
15413,BA:K 010000-001,BA:K 010000-001:ba_synonym:betonbau,K 010000-001,,Betonbau,ba_synonym
15414,BA:K 010000-001,BA:K 010000-001:ba_synonym:fertigteilherstellu...,K 010000-001,,Fertigteilherstellung (Beton),ba_synonym


BA Leaf-Skills vs. Groups Results: This Block 10 is for testing purposes only. The breakdown is ambiguous (groups are not clearly defined, many are “unknown”), so it is not suitable for potential filtering in the profile extension. Its original form will therefore continue to be used.

## 8. Save + Checks

In [21]:
OUT_VOCAB_CSV = DATA_PROCESSED_EXTERNAL / "skills_vocab_with_linkedin.csv"
OUT_VOCAB_PARQUET = DATA_PROCESSED_EXTERNAL / "skills_vocab_with_linkedin_with_skill_id.parquet"
OUT_CONCEPTS = DATA_PROCESSED_EXTERNAL / "skills_concepts.parquet"
OUT_VOCAB_PARQUET_FILTERED = DATA_PROCESSED_EXTERNAL / "skills_vocab_with_linkedin_with_skill_id_filtered.parquet"
OUT_CONCEPTS_FILTERED = DATA_PROCESSED_EXTERNAL / "skills_concepts_filtered.parquet"

# Save full
df_vocab_extended.to_csv(OUT_VOCAB_CSV, index=False, encoding="utf-8")
df_vocab_extended.to_parquet(OUT_VOCAB_PARQUET, index=False)
skills_concepts_full.to_parquet(OUT_CONCEPTS, index=False)

# Save filtered (For greater precision in 10b)
df_vocab_extended_filtered.to_parquet(OUT_VOCAB_PARQUET_FILTERED, index=False)
skills_concepts_filtered.to_parquet(OUT_CONCEPTS_FILTERED, index=False)

print("Saved:")
print("-", OUT_VOCAB_CSV)
print("-", OUT_VOCAB_PARQUET)
print("-", OUT_CONCEPTS)
print("-", OUT_VOCAB_PARQUET_FILTERED)
print("-", OUT_CONCEPTS_FILTERED)

# Output checks
assert df_vocab_extended["skill_id"].isna().sum() == 0
assert df_vocab_extended["label_norm"].isna().sum() == 0
assert df_vocab_extended["vocab_id"].isna().sum() == 0
print("OK: full vocab checks passed")

assert df_vocab_extended_filtered["skill_id"].isna().sum() == 0
print("OK: filtered vocab checks passed")

Saved:
- C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\skills_vocab_with_linkedin.csv
- C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\skills_vocab_with_linkedin_with_skill_id.parquet
- C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\skills_concepts.parquet
- C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\skills_vocab_with_linkedin_with_skill_id_filtered.parquet
- C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\skills_concepts_filtered.parquet
OK: full vocab checks passed
OK: filtered vocab checks passed


In [22]:
# Preview
print("Sample vocab rows:")
cols_vocab_preview = [c for c in ["vocab_id","skill_id","label","label_type","source_system","lang"] if c in df_vocab_extended.columns]
display(df_vocab_extended.sample(10, random_state=42)[cols_vocab_preview])

print("\nSample concepts:")
cols_concepts_preview = [c for c in ["skill_id","display_label_fallback","display_label"] if c in skills_concepts.columns]
display(skills_concepts.sample(10, random_state=42)[cols_concepts_preview])

Sample vocab rows:


,vocab_id,skill_id,label,label_type,source_system,lang
85022,BA:K 131007-000:ba_synonym_tech:aussenstuckfas...,BA:K 131007-000,AUSSENSTUCKFASSADENSTUCK,ba_synonym_tech,BA,de
43174,BA:K 070100-103:ba_synonym:cad-programm vicado,BA:K 070100-103,CAD-Programm ViCADo,ba_synonym,BA,de
102600,LINKEDIN:ebxml,LINKEDIN:ebxml,EbXML,linkedin_skill,LINKEDIN,en
74103,BA:K 0904-028:ba_synonym:drogenabhängige beraten,BA:K 0904-028,Drogenabhängige beraten,ba_synonym,BA,de
95336,LINKEDIN:ccea,LINKEDIN:ccea,CCEA,linkedin_skill,LINKEDIN,en
113762,LINKEDIN:organizational consulting,LINKEDIN:organizational consulting,Organizational Consulting,linkedin_skill,LINKEDIN,en
117442,LINKEDIN:raiser's edge,LINKEDIN:raiser's edge,Raiser's Edge,linkedin_skill,LINKEDIN,en
74317,BA:K 100000-012:ba_synonym:feuerschlucken,BA:K 100000-012,Feuerschlucken,ba_synonym,BA,de
70133,BA:K 090206-031:ba_synonym_tech:molekularbiolo...,BA:K 090206-031,MOLEKULARBIOLOGISCHEVERFAHREN,ba_synonym_tech,BA,de
16051,BA:K 010100-022:ba_synonym:kollektionen entwic...,BA:K 010100-022,Kollektionen entwickeln,ba_synonym,BA,de



Sample concepts:


,skill_id,display_label_fallback,display_label
22645,ESCO:http://data.europa.eu/esco/skill/ef822959...,provide anaesthetics to animals,provide anaesthetics to animals
51296,LINKEDIN:sctp,SCTP,NaN
12928,ESCO:http://data.europa.eu/esco/skill/3bf7e0ee...,manage mine site data,manage mine site data
48671,LINKEDIN:promotional marketing,Promotional Marketing,NaN
10415,ESCO:http://data.europa.eu/esco/skill/0e1fe34b...,manage schedule of tasks,manage schedule of tasks
13371,ESCO:http://data.europa.eu/esco/skill/442be689...,parts of a surface grinding machine,parts of a surface grinding machine
55429,LINKEDIN:tonality pro,Tonality Pro,NaN
49395,LINKEDIN:radioss,Radioss,NaN
19174,ESCO:http://data.europa.eu/esco/skill/ae6d655d...,safeguard biodiversity,safeguard biodiversity
13701,ESCO:http://data.europa.eu/esco/skill/4a752bd2...,community education,community education


Final Output-Checks:

In [23]:
print("\nOutput-Checks:")
print("Rows total:", len(df_vocab_extended))
if "source_system" in df_vocab_extended.columns:
    print(df_vocab_extended["source_system"].value_counts())

# Hard Quality Checks
for col in ["skill_id","label_norm","vocab_id"]:
    if col in df_vocab_extended.columns:
        assert df_vocab_extended[col].isna().sum() == 0, f"Missing values in {col}"

if "vocab_id" in df_vocab_extended.columns: # vocab_id must be unique
    dup = df_vocab_extended["vocab_id"].duplicated().sum()
    print("Duplicate vocab_id:", dup)
    assert dup == 0, "vocab_id is not unique!"

print("OK: vocab checks passed")

if "df_vocab_extended_filtered" in globals(): # filtered vocab
    for col in ["skill_id","label_norm","vocab_id"]:
        if col in df_vocab_extended_filtered.columns:
            assert df_vocab_extended_filtered[col].isna().sum() == 0, f"Missing values in filtered {col}"
    dup_f = df_vocab_extended_filtered["vocab_id"].duplicated().sum()
    assert dup_f == 0, "filtered vocab_id is not unique!"
    print("OK: filtered vocab checks passed")


Output-Checks:
Rows total: 126277
source_system
BA          77464
LINKEDIN    34874
ESCO        13939
Name: count, dtype: int64
Duplicate vocab_id: 0
OK: vocab checks passed
OK: filtered vocab checks passed


Output files:
- `skills_vocab_with_linkedin_with_skill_id.parquet`: complete vocabulary (ESCO+BA + LinkedIn), including `skill_id` and `label_norm`.
- `skills_vocab_with_linkedin_with_skill_id_filtered.parquet`: same as above, but with LinkedIn noise (very short/character-based entries) removed (recommended for Notebook 10b).
- `skills_concepts.parquet` / `skills_concepts_filtered.parquet`: one table each with 1 row per `skill_id` for display (`display_label`) and for later aggregation/evaluation.

## 9. Mini Skill Vocabulary from data/raw_external/csv/Resume dataset (structured)/06_skills.csv

Optional addition: `Resume dataset (structured): 06_skills.csv` (approx. 227,000 skills, Kaggle - Ganesh, https://www.kaggle.com/datasets/suriyaganesh/resume-dataset-structured/data?select=01_people.csv) will likely not be included (very large number, performance and noise risk). As an optional addition, the file is saved as an alternative export variant in the same format.

In [24]:
# Mini Vocabulary from Resume 06_skills.csv
def norm_label(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s.strip()

# Expected column: skill
df_06 = pd.read_csv(RESUME06_SKILLS_PATH)
assert "skill" in df_06.columns, f"Expected column 'skill' in {RESUME06_SKILLS_PATH}"

df_06["label"] = df_06["skill"].astype(str).str.strip()
df_06["label_norm"] = df_06["label"].map(norm_label)
df_06 = df_06[df_06["label_norm"] != ""].drop_duplicates(subset=["label_norm"]).reset_index(drop=True) # Remove duplicates

df_resume_vocab = pd.DataFrame({
    "source_system": "RESUME",
    "source_id": "resume_06_skills",
    "lang": "en",
    "label_type": "resume_skill",
    "label": df_06["label"],
    "label_norm": df_06["label_norm"],
    "is_noise": False,
})

# Stable IDs in Mini-Vocabs
df_resume_vocab["skill_id"] = "RESUME:" + df_resume_vocab["label_norm"]
df_resume_vocab["vocab_id"] = "RESUME:" + df_resume_vocab["label_norm"]

OUT_RESUME_VOCAB = DATA_PROCESSED_EXTERNAL / "skills_vocab_resume_06_skills.parquet"
df_resume_vocab.to_parquet(OUT_RESUME_VOCAB, index=False)
print("Saved optional resume vocab:", OUT_RESUME_VOCAB)
print("Rows:", len(df_resume_vocab))
display(df_resume_vocab.head(10))

Saved optional resume vocab: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\skills_vocab_resume_06_skills.parquet
Rows: 204192


,source_system,source_id,lang,label_type,label,label_norm,is_noise,skill_id,vocab_id
0,RESUME,resume_06_skills,en,resume_skill,Mongo DB-3.2,mongo db-3.2,False,RESUME:mongo db-3.2,RESUME:mongo db-3.2
1,RESUME,resume_06_skills,en,resume_skill,JNDI LDAP,jndi ldap,False,RESUME:jndi ldap,RESUME:jndi ldap
2,RESUME,resume_06_skills,en,resume_skill,Stored Procedures,stored procedures,False,RESUME:stored procedures,RESUME:stored procedures
3,RESUME,resume_06_skills,en,resume_skill,Perform ad-hoc analysis,perform ad-hoc analysis,False,RESUME:perform ad-hoc analysis,RESUME:perform ad-hoc analysis
4,RESUME,resume_06_skills,en,resume_skill,Monitored and resolved flight crew legality is...,monitored and resolved flight crew legality is...,False,RESUME:monitored and resolved flight crew lega...,RESUME:monitored and resolved flight crew lega...
5,RESUME,resume_06_skills,en,resume_skill,Dashboard System,dashboard system,False,RESUME:dashboard system,RESUME:dashboard system
6,RESUME,resume_06_skills,en,resume_skill,Operate unit level computers,operate unit level computers,False,RESUME:operate unit level computers,RESUME:operate unit level computers
7,RESUME,resume_06_skills,en,resume_skill,Setting Up Test Environments,setting up test environments,False,RESUME:setting up test environments,RESUME:setting up test environments
8,RESUME,resume_06_skills,en,resume_skill,Write test cases,write test cases,False,RESUME:write test cases,RESUME:write test cases
9,RESUME,resume_06_skills,en,resume_skill,Server resources,server resources,False,RESUME:server resources,RESUME:server resources


# Conclusion 10a: Skill Dictionary Expansion for Method 1.1 (Rule-/Lexicon-Based Profile Expansion)

In this notebook, the skill dictionary for Method 1.1 was expanded:
- Starting point: ESCO skills + BA competencies (see Notebooks 01–04, 07).
- Expansion: LinkedIn/Employment Skills list (analogous to the multi-dictionary approach in Cenikj et al. (2021)).
   - Result: `skills_vocab_with_linkedin.csv` with > 120,000 vocabulary entries.
   - Concept ID level (central to 1.1): Each vocabulary row was assigned a stable `skill_id`. This allows subsequent extraction and aggregation to run consistently via `skill_id` (rather than via `vocab_id`).
   - Concept table for display/joins: `skills_concepts.parquet` contains one row per `skill_id` with `display_label` (preferred, otherwise fallback).
   - Quality checks: No missing `skill_id` or `label_norm` values; `vocab_id` is unique (no duplicates); LinkedIn entries with a higher noise ratio have been identified and can be optionally filtered in Notebook 10b.
   - For Notebook 10_b: Use `skill_id` as the join key (concept level). `vocab_id` remains purely a row ID for the label lexicon. Therefore, in addition to the CSV, a Parquet version containing `skill_id` and `label_norm` is also stored. In this case, `skills_vocab_with_linkedin_with_skill_id.parquet` and `skills_concepts.parquet` are loaded by default.